In [ ]:
# Environment Setup (Local TITAN X / Colab T4)
# V9 OPTIMAL - Auto-detects environment and GPU capacity

import os
import gc

# Detect environment
try:
 from google.colab import drive
 drive.mount('/content/drive')
 print(" Google Drive mounted - Running on Colab")
 IN_COLAB = True
except:
 print(" Running on Local Machine")
 IN_COLAB = False

# Install packages based on environment
if IN_COLAB:!pip install pydicom -q
else:
 # Local: use existing venv packages, just ensure pydicom!pip install pydicom -q 2>/dev/null || true

import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from PIL import Image
import warnings
import cv2
from tqdm import tqdm
import json
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Metrics & visualization
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Detect GPU and set memory growth
gpus = tf.config.list_physical_devices('GPU')
GPU_NAME = "CPU"
GPU_MEMORY_GB = 0

if gpus:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 
 # Get GPU info
 gpu_details = tf.config.experimental.get_device_details(gpus[0])
 GPU_NAME = gpu_details.get('device_name', 'Unknown GPU')
 
 # Estimate memory (TITAN X = 12GB, T4 = 16GB)
 if 'TITAN' in GPU_NAME.upper():
 GPU_MEMORY_GB = 12
 GPU_TYPE = "TITAN_X"
 elif 'T4' in GPU_NAME.upper():
 GPU_MEMORY_GB = 16
 GPU_TYPE = "T4"
 elif 'A100' in GPU_NAME.upper():
 GPU_MEMORY_GB = 40
 GPU_TYPE = "A100"
 else:
 GPU_MEMORY_GB = 8 # Conservative default
 GPU_TYPE = "OTHER"

print(f"\n{'='*60}")
print(f" ENVIRONMENT CONFIGURATION")
print(f"{'='*60}")
print(f" Platform: {'Google Colab' if IN_COLAB else 'Local Machine'}")
print(f" TensorFlow: {tf.__version__}")
print(f" NumPy: {np.__version__}")
print(f" GPU: {GPU_NAME}")
print(f" GPU Memory: ~{GPU_MEMORY_GB} GB")
print(f"{'='*60}")

In [ ]:
# V9 OPTIMAL CONFIGURATION (Auto-GPU-Optimized)
# Automatically adjusts settings based on detected GPU capacity
# TITAN X (12GB): batch=8, dense=1024, single model
# T4 (16GB): batch=16, dense=2048, ensemble

class Config:
 """
 V9 OPTIMAL Configuration - Auto-GPU-Optimized
 
 GPU PROFILES:
 - TITAN X (12GB): Conservative settings to avoid OOM
 - T4 (16GB): Full paper spec settings
 - A100 (40GB+): Maximum settings
 """
 
 def __init__(self, gpu_memory_gb=12):
 # Store GPU info
 self.GPU_MEMORY_GB = gpu_memory_gb
 
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False
 
 # Set paths based on environment
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v9'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v9'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v9'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v9'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v9'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v9'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 
 # Data files
 self.CALC_TRAIN_CSV = self.CSV_PATH / 'calc_case_description_train_set.csv'
 self.CALC_TEST_CSV = self.CSV_PATH / 'calc_case_description_test_set.csv'
 self.MASS_TRAIN_CSV = self.CSV_PATH / 'mass_case_description_train_set.csv'
 self.MASS_TEST_CSV = self.CSV_PATH / 'mass_case_description_test_set.csv'
 
 # IMAGE CONFIGURATION
 self.IMG_SIZE = (224, 224) # Paper spec - proven stable
 self.IMG_CHANNELS = 3
 
 # CLAHE PREPROCESSING
 self.USE_CLAHE = True
 self.CLAHE_CLIP_LIMIT = 2.0
 self.CLAHE_TILE_SIZE = (8, 8)
 
 # GPU-OPTIMIZED SETTINGS
 if gpu_memory_gb >= 16:
 # T4 / V100 / A100: Full settings
 self.BATCH_SIZE = 16
 self.DENSE_UNITS = 2048
 self.NUM_ENSEMBLE_MODELS = 3
 self.USE_MIXED_PRECISION = True
 self.GPU_PROFILE = "HIGH_MEMORY"
 elif gpu_memory_gb >= 12:
 # TITAN X / RTX 3080: Conservative settings
 self.BATCH_SIZE = 8
 self.DENSE_UNITS = 1024
 self.NUM_ENSEMBLE_MODELS = 1 # Single model to avoid OOM
 self.USE_MIXED_PRECISION = False # FP16 causes cuDNN issues on older GPUs
 self.GPU_PROFILE = "TITAN_X"
 else:
 # Low memory: Minimal settings
 self.BATCH_SIZE = 4
 self.DENSE_UNITS = 512
 self.NUM_ENSEMBLE_MODELS = 1
 self.USE_MIXED_PRECISION = False
 self.GPU_PROFILE = "LOW_MEMORY"
 
 self.ENSEMBLE_SEEDS = [42, 123, 456][:self.NUM_ENSEMBLE_MODELS]
 
 # TEST-TIME AUGMENTATION
 self.USE_TTA = True
 self.TTA_AUGMENTATIONS = 4
 
 # TRAINING PARAMETERS
 self.STAGE1_EPOCHS = 25
 self.STAGE1_LR = 1e-3
 
 self.STAGE2_EPOCHS = 40
 self.STAGE2_LR = 5e-4
 
 self.STAGE3_EPOCHS = 80
 self.STAGE3_LR = 1e-4
 
 self.USE_LR_WARMUP = True
 self.WARMUP_EPOCHS = 5
 self.WARMUP_START_LR = 1e-7
 
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15
 
 # REGULARIZATION
 self.L2_REGULARIZATION = 1e-4
 self.DROPOUT_RATE = 0.25
 self.LABEL_SMOOTHING = 0.1
 self.GRADIENT_CLIPNORM = 1.0
 
 # CLASS WEIGHTING
 self.USE_CLASS_WEIGHT = True
 self.MALIGNANT_WEIGHT_MULTIPLIER = 2.5
 
 # FOCAL LOSS
 self.USE_FOCAL_LOSS = True
 self.FOCAL_LOSS_ALPHA = 0.70
 self.FOCAL_LOSS_GAMMA = 2.0
 
 # ARCHITECTURE
 self.FREEZE_STAGE2 = 300
 self.FREEZE_STAGE3 = 100
 
 # CALLBACKS
 self.EARLY_STOP_PATIENCE = 20
 self.LR_REDUCE_PATIENCE = 8
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 15
 self.MIN_EPOCHS_STAGE2 = 25
 self.MIN_EPOCHS_STAGE3 = 40
 
 # PERFORMANCE OPTIMIZATION
 self.PREFETCH_BUFFER = tf.data.AUTOTUNE
 self.SHUFFLE_BUFFER = 1000
 self.NUM_PARALLEL_CALLS = tf.data.AUTOTUNE
 self.FORCE_CACHE_REBUILD = False
 
 self._create_dirs()
 
 def _create_dirs(self):
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH, self.CACHE_PATH]:
 path.mkdir(parents=True, exist_ok=True)
 
 def print_config(self):
 print("\n" + "="*70)
 print(" V9 OPTIMAL - GPU-OPTIMIZED CONFIGURATION")
 print("="*70)
 
 print(f"\n GPU Profile: {self.GPU_PROFILE} ({self.GPU_MEMORY_GB}GB)")
 print(f" Batch Size: {self.BATCH_SIZE}")
 print(f" Dense Units: {self.DENSE_UNITS}")
 print(f" Ensemble: {self.NUM_ENSEMBLE_MODELS} model(s)")
 print(f" Mixed Precision:{self.USE_MIXED_PRECISION}")
 
 print(f"\n Environment: {'Google Colab' if self.IN_COLAB else 'Local Machine'}")
 print(f" Cache: {self.CACHE_PATH}")
 print(f" Checkpoints: {self.CHECKPOINT_PATH}")
 
 print(f"\n Training Strategy:")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs @ LR={self.STAGE1_LR}")
 print(f" Stage 2: {self.STAGE2_EPOCHS} epochs @ LR={self.STAGE2_LR}")
 print(f" Stage 3: {self.STAGE3_EPOCHS} epochs @ LR={self.STAGE3_LR}")
 total_epochs = (self.STAGE1_EPOCHS + self.STAGE2_EPOCHS + self.STAGE3_EPOCHS) * self.NUM_ENSEMBLE_MODELS
 print(f" Total: {total_epochs} epochs")
 
 # Check if cache exists
 cache_files = ['train_cache_v9.npz', 'val_cache_v9.npz', 'test_cache_v9.npz']
 cache_exists = all((self.CACHE_PATH / f).exists() for f in cache_files)
 if cache_exists:
 print(f"\n PREPROCESSED CACHE FOUND - Fast loading enabled!")
 else:
 print(f"\n⏳ No cache found - First run will preprocess data")
 
 print("="*70 + "\n")

# Initialize configuration with detected GPU memory
config = Config(gpu_memory_gb=GPU_MEMORY_GB)
config.print_config()

# Configure GPU
if gpus:
 if config.USE_MIXED_PRECISION:
 try:
 policy = tf.keras.mixed_precision.Policy('mixed_float16')
 tf.keras.mixed_precision.set_global_policy(policy)
 print(" Mixed precision (FP16) enabled")
 except Exception as e:
 print(f" Mixed precision not available: {e}")
 config.USE_MIXED_PRECISION = False
 else:
 print(" Using FP32 (mixed precision disabled for GPU compatibility)")

In [ ]:
# Verify Data Paths & Cache Status

print(" Verifying data paths and cache status...")
print(f" Environment: {'Colab' if config.IN_COLAB else 'Local'}")
print(f" Base path: {config.BASE_PATH}")

# Check paths
paths_to_check = [
 ('DICOM folder', config.DICOM_PATH),
 ('CSV folder', config.CSV_PATH),
 ('Cache folder', config.CACHE_PATH),
]

all_ok = True
for name, path in paths_to_check:
 if path.exists():
 print(f" {name}")
 else:
 print(f" {name} NOT FOUND: {path}")
 all_ok = False

# Check cache status
cache_files = {
 'train_cache_v9.npz': config.CACHE_PATH / 'train_cache_v9.npz',
 'val_cache_v9.npz': config.CACHE_PATH / 'val_cache_v9.npz',
 'test_cache_v9.npz': config.CACHE_PATH / 'test_cache_v9.npz',
}

print(f"\n Cache Status:")
cache_ready = True
total_cache_mb = 0
for name, path in cache_files.items():
 if path.exists():
 size_mb = path.stat().st_size / (1024 * 1024)
 total_cache_mb += size_mb
 print(f" {name} ({size_mb:.1f} MB)")
 else:
 print(f" {name} - not found")
 cache_ready = False

if cache_ready:
 print(f"\n CACHE READY ({total_cache_mb:.1f} MB total)")
 print(" Preprocessing will be skipped - instant data loading!")
else:
 print(f"\n⏳ Cache not found - first run will preprocess all images")
 print(" This takes ~5-10 minutes but only happens once.")

if all_ok:
 print(f"\n Ready to proceed!")
else:
 print(f"\n Some paths missing - check configuration")

---
## Phase 1: Data Pre-processing (Conducted ONCE)
Following the research paper pipeline: Load → Split → Encode → Normalize → Cache

In [ ]:
# CLAHE-Enhanced DICOM Loading

def find_dicom_file(case_dir_name, base_path):
 """Find ROI DICOM file in case directory."""
 case_path = Path(base_path) / case_dir_name
 
 if not case_path.exists():
 return None
 
 try:
 dcm_files = list(case_path.rglob("*.dcm"))
 if not dcm_files:
 return None
 dcm_files = sorted(dcm_files, key=lambda x: x.name)
 return dcm_files[-1] if len(dcm_files) >= 2 else dcm_files[0]
 except:
 return None


def apply_clahe(image, clip_limit=2.0, tile_size=(8, 8)):
 """
 Apply CLAHE (Contrast Limited Adaptive Histogram Equalization).
 
 CLAHE is critical for mammography:
 - Enhances local contrast in dense breast tissue
 - Preserves fine details (microcalcifications)
 - Doesn't over-amplify noise like global HE
 """
 img_8bit = (image * 255).astype(np.uint8)
 clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
 enhanced = clahe.apply(img_8bit)
 return enhanced.astype(np.float32) / 255.0


def load_and_preprocess_dicom(dcm_path, config):
 """
 V9: Load DICOM with CLAHE preprocessing.
 
 Pipeline:
 1. Load DICOM → pixel array
 2. Apply rescale slope/intercept
 3. Apply VOI LUT or window/level
 4. Handle photometric interpretation
 5. Percentile normalization [1%, 99%]
 6. CLAHE enhancement
 7. Resize to target size
 8. Convert to RGB + DenseNet preprocessing
 """
 try:
 dcm = pydicom.dcmread(str(dcm_path))
 img = dcm.pixel_array.astype(np.float32)
 
 if img.size == 0:
 return None
 
 # Apply Rescale Slope and Intercept
 rescale_slope = float(getattr(dcm, 'RescaleSlope', 1.0))
 rescale_intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
 if rescale_slope!= 1.0 or rescale_intercept!= 0.0:
 img = img * rescale_slope + rescale_intercept
 
 # Apply VOI LUT or Window/Level
 voi_applied = False
 if hasattr(dcm, 'VOILUTSequence') and dcm.VOILUTSequence:
 try:
 from pydicom.pixel_data_handlers.util import apply_voi_lut
 img = apply_voi_lut(img.astype(np.float64), dcm, index=0).astype(np.float32)
 voi_applied = True
 except:
 pass
 
 if not voi_applied:
 window_center = getattr(dcm, 'WindowCenter', None)
 window_width = getattr(dcm, 'WindowWidth', None)
 
 if window_center is not None and window_width is not None:
 wc = float(window_center[0]) if hasattr(window_center, '__iter__') else float(window_center)
 ww = float(window_width[0]) if hasattr(window_width, '__iter__') else float(window_width)
 
 if ww > 0:
 img_min = wc - ww / 2
 img_max = wc + ww / 2
 img = np.clip(img, img_min, img_max)
 img = (img - img_min) / (img_max - img_min)
 voi_applied = True
 
 # Handle Photometric Interpretation
 photometric = getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
 if 'MONOCHROME1' in str(photometric):
 img = 1.0 - img if voi_applied else np.max(img) - img
 
 # Percentile normalization if VOI wasn't applied
 if not voi_applied:
 p1, p99 = np.percentile(img, [1, 99])
 if p99 > p1:
 img = (img - p1) / (p99 - p1)
 else:
 img_min, img_max = img.min(), img.max()
 img = (img - img_min) / (img_max - img_min + 1e-8)
 
 img = np.clip(img, 0.0, 1.0)
 
 # Apply CLAHE (before resizing to preserve detail)
 if config.USE_CLAHE:
 img = apply_clahe(img, config.CLAHE_CLIP_LIMIT, config.CLAHE_TILE_SIZE)
 
 # Resize to target size
 img = cv2.resize(img, config.IMG_SIZE, interpolation=cv2.INTER_LANCZOS4)
 
 # Convert to RGB and apply DenseNet preprocessing
 img_rgb = np.stack([img, img, img], axis=-1)
 img_rgb = img_rgb * 255.0
 img_rgb = densenet_preprocess(img_rgb)
 
 return img_rgb.astype(np.float32)
 except Exception as e:
 return None

print(" V9 CLAHE-enhanced DICOM loading utilities defined")
print(f" Resolution: {config.IMG_SIZE}")
print(f" CLAHE: clip={config.CLAHE_CLIP_LIMIT}, tiles={config.CLAHE_TILE_SIZE}")

In [ ]:
# Load Dataset Metadata

def extract_case_dir_from_path(path_str):
 """Extract case directory name from CSV path."""
 if pd.isna(path_str) or path_str == '':
 return None
 parts = str(path_str).split('/')
 return parts[0] if len(parts) >= 1 else None


def load_cbis_ddsm_metadata(config):
 """Load CBIS-DDSM dataset metadata."""
 
 print("\n" + "="*70)
 print(" LOADING CBIS-DDSM METADATA")
 print("="*70)
 
 dfs = []
 csv_files = [
 (config.CALC_TRAIN_CSV, 'calc', 'train'),
 (config.CALC_TEST_CSV, 'calc', 'test'),
 (config.MASS_TRAIN_CSV, 'mass', 'train'),
 (config.MASS_TEST_CSV, 'mass', 'test'),
 ]
 
 for csv_path, lesion_type, split in csv_files:
 if csv_path.exists():
 df = pd.read_csv(csv_path)
 df['lesion_type'] = lesion_type
 df['original_split'] = split
 dfs.append(df)
 print(f" Loaded {csv_path.name}: {len(df)} records")
 else:
 print(f" Missing {csv_path.name}")
 
 if not dfs:
 raise FileNotFoundError("No CSV files found!")
 
 combined_df = pd.concat(dfs, ignore_index=True)
 print(f"\n Total records: {len(combined_df)}")
 
 # Encode labels (Binary: Benign=0, Malignant=1)
 combined_df['label'] = combined_df['pathology'].apply(
 lambda x: 1 if 'MALIGNANT' in str(x).upper() else 0
 )
 
 # Find path column
 path_col = None
 for col in ['cropped image file path', 'ROI mask file path', 'image file path']:
 if col in combined_df.columns:
 path_col = col
 break
 
 if path_col is None:
 raise ValueError("Could not find image path column!")
 
 print(f" Using path column: '{path_col}'")
 
 # Extract case_dir
 combined_df['case_dir'] = combined_df[path_col].apply(extract_case_dir_from_path)
 combined_df = combined_df[combined_df['case_dir'].notna()].copy()
 
 # Validate DICOM files exist
 print(f"\n Validating DICOM files...")
 
 valid_samples = []
 for _, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc="Validating"):
 dcm_path = find_dicom_file(row['case_dir'], config.DICOM_PATH)
 if dcm_path:
 valid_samples.append({
 'case_dir': row['case_dir'],
 'dcm_path': str(dcm_path),
 'label': row['label'],
 'lesion_type': row['lesion_type'],
 'pathology': row['pathology']
 })
 
 samples_df = pd.DataFrame(valid_samples)
 
 print(f"\n Valid samples: {len(samples_df)}")
 print(f"\n Class Distribution:")
 print(f" Benign (0): {(samples_df['label'] == 0).sum()} ({(samples_df['label'] == 0).mean()*100:.1f}%)")
 print(f" Malignant (1): {(samples_df['label'] == 1).sum()} ({(samples_df['label'] == 1).mean()*100:.1f}%)")
 
 return samples_df

# Load metadata
samples_df = load_cbis_ddsm_metadata(config)
print("="*70)

In [ ]:
# Stratified Data Splitting (Paper spec: 70-15-15)

print("\n" + "="*70)
print(" STRATIFIED DATA SPLITTING (70-15-15)")
print("="*70)

# First split: train vs (val + test)
train_df, temp_df = train_test_split(
 samples_df,
 test_size=(config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=samples_df['label'],
 random_state=42
)

# Second split: val vs test
val_df, test_df = train_test_split(
 temp_df,
 test_size=config.TEST_SPLIT / (config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=temp_df['label'],
 random_state=42
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n Split Results:")
print(f" Training: {len(train_df)} samples ({len(train_df)/len(samples_df)*100:.1f}%)")
print(f" Validation: {len(val_df)} samples ({len(val_df)/len(samples_df)*100:.1f}%)")
print(f" Test: {len(test_df)} samples ({len(test_df)/len(samples_df)*100:.1f}%)")

print(f"\n Class Distribution per Split:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
 n_benign = (df['label'] == 0).sum()
 n_malignant = (df['label'] == 1).sum()
 ratio = n_malignant / n_benign if n_benign > 0 else 0
 print(f" {name}: Benign={n_benign}, Malignant={n_malignant} (ratio: 1:{1/ratio:.2f})")

print("="*70)

In [ ]:
# Preprocessing & Caching (Conducted ONCE)

def preprocess_and_cache(df, config, split_name):
 """
 V9: Preprocess all images with CLAHE and cache to disk.
 
 First run: 5-10 minutes
 Future runs: Loads instantly from cache!
 """
 cache_file = config.CACHE_PATH / f'{split_name}_cache_v9.npz'
 
 # Check cache
 if cache_file.exists() and not config.FORCE_CACHE_REBUILD:
 print(f"\n Loading {split_name} from cache...")
 data = np.load(cache_file)
 images = data['images']
 labels = data['labels']
 print(f" Loaded {len(images)} images instantly!")
 return images, labels
 
 # First time: preprocess all images
 print(f"\n{'='*70}")
 print(f"⏳ PREPROCESSING {split_name.upper()} (CLAHE + {config.IMG_SIZE})")
 print(f"{'='*70}")
 print(f" First run only - future runs load from cache\n")
 
 images = []
 labels = []
 failed = 0
 
 for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
 img = load_and_preprocess_dicom(row['dcm_path'], config)
 if img is not None:
 images.append(img)
 labels.append(row['label'])
 else:
 failed += 1
 
 images = np.array(images, dtype=np.float32)
 labels = np.array(labels, dtype=np.float32)
 
 print(f"\n Preprocessed {len(images)} images")
 if failed > 0:
 print(f" Failed: {failed} images")
 
 # Save cache
 print(f" Saving cache...")
 np.savez_compressed(cache_file, images=images, labels=labels)
 cache_size_mb = cache_file.stat().st_size / (1024 * 1024)
 print(f" Saved to {cache_file.name} ({cache_size_mb:.1f} MB)")
 
 return images, labels


# Preprocess and cache all splits
print("\n" + "="*70)
print(" V9 PREPROCESSING & CACHING")
print("="*70)

train_images, train_labels = preprocess_and_cache(train_df, config, 'train')
val_images, val_labels = preprocess_and_cache(val_df, config, 'val')
test_images, test_labels = preprocess_and_cache(test_df, config, 'test')

print(f"\n All data loaded into memory:")
print(f" Train: {train_images.shape} ({train_images.nbytes / 1e6:.1f} MB)")
print(f" Val: {val_images.shape}")
print(f" Test: {test_images.shape}")

# Disable forced rebuild after first run
config.FORCE_CACHE_REBUILD = False

In [ ]:
# tf.data Pipeline with GPU Augmentation
# FIX: Use from_generator instead of from_tensor_slices
# to avoid GPU memory error when copying large arrays

@tf.function
def augment_batch(images):
 """V9 GPU-accelerated augmentation."""
 # Random horizontal flip
 images = tf.image.random_flip_left_right(images)
 # Random vertical flip
 images = tf.image.random_flip_up_down(images)
 # Random brightness (±10%)
 images = tf.image.random_brightness(images, max_delta=0.1)
 # Random contrast (±10%)
 images = tf.image.random_contrast(images, lower=0.9, upper=1.1)
 return images


def create_data_generator(images, labels, shuffle=False, seed=None):
 """Generator function that yields (image, label) pairs."""
 indices = np.arange(len(images))
 if shuffle:
 rng = np.random.default_rng(seed)
 rng.shuffle(indices)
 
 for idx in indices:
 yield images[idx], labels[idx]


def create_dataset(images, labels, config, training=True, shuffle_seed=None):
 """
 Create optimized tf.data.Dataset using from_generator.
 
 FIX: from_tensor_slices caused GPU memory error:
 "Failed copying input tensor from CPU to GPU"
 
 from_generator streams data instead of copying all at once.
 """
 output_signature = (
 tf.TensorSpec(shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3), dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 
 dataset = tf.data.Dataset.from_generator(
 lambda: create_data_generator(images, labels, shuffle=training, seed=shuffle_seed),
 output_signature=output_signature
 )
 
 # Shuffle buffer for additional randomization during training
 if training:
 dataset = dataset.shuffle(config.SHUFFLE_BUFFER)
 
 dataset = dataset.batch(config.BATCH_SIZE)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset


# Create datasets
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print("\n V9 tf.data pipelines created (using from_generator)")
print(f" Batch size: {config.BATCH_SIZE}")
print(f" Steps per epoch: {len(train_images) // config.BATCH_SIZE}")
print(f" Streams data to GPU (avoids memory overflow)")

In [ ]:
# Compute Class Weights

balanced_weights = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=train_labels
)

config.CLASS_WEIGHTS = {
 0: balanced_weights[0],
 1: balanced_weights[1] * config.MALIGNANT_WEIGHT_MULTIPLIER
}

print("\n" + "="*70)
print(" V9 CLASS WEIGHTING")
print("="*70)
print(f"\n Training Class Distribution:")
print(f" Benign (0): {int((train_labels == 0).sum())} ({(train_labels == 0).mean()*100:.1f}%)")
print(f" Malignant (1): {int((train_labels == 1).sum())} ({(train_labels == 1).mean()*100:.1f}%)")
print(f"\n Class Weights (with {config.MALIGNANT_WEIGHT_MULTIPLIER}x malignant boost):")
print(f" Benign (0): {config.CLASS_WEIGHTS[0]:.3f}")
print(f" Malignant (1): {config.CLASS_WEIGHTS[1]:.3f}")
print("="*70)

---
## Phase 2: Model Training
Following the paper: Create Model → Compile → Train → Persist

In [ ]:
# Focal Loss Definition

@tf.function
def focal_loss_fn(y_true, y_pred, gamma=2.0, alpha=0.7):
 """Focal loss for handling class imbalance."""
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 y_true = tf.cast(y_true, tf.float32)
 
 # Cross entropy
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 
 # Focal weight
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, gamma)
 
 # Alpha weight
 alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 return tf.reduce_mean(alpha_weight * focal_weight * ce)


def get_focal_loss(gamma, alpha):
 """Return focal loss function for model.compile()."""
 def loss_fn(y_true, y_pred):
 return focal_loss_fn(y_true, y_pred, gamma, alpha)
 return loss_fn

print(" Focal Loss defined")
print(f" α = {config.FOCAL_LOSS_ALPHA} (minority class weight)")
print(f" γ = {config.FOCAL_LOSS_GAMMA} (focusing parameter)")

In [ ]:
# V9 Model Builder (Paper Spec: 2048 units)

def build_model(config, seed=42):
 """
 Build V9 model following research paper architecture.
 
 Paper Spec:
 - DenseNet121 base (ImageNet pretrained)
 - Global Average Pooling
 - BatchNorm → Dense(2048, ReLU, L2) → BatchNorm → Dropout → Output
 """
 # Set seed for reproducibility
 tf.random.set_seed(seed)
 np.random.seed(seed)
 
 print(f"\n Building V9 Model (seed={seed})")
 
 # Base model
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3),
 pooling=None
 )
 base_model.trainable = False
 
 # Custom classification head (Paper spec)
 x = base_model.output
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.BatchNormalization(name='bn_1')(x)
 
 # Paper spec: 2048 units (used 1024 before!)
 x = layers.Dense(
 config.DENSE_UNITS, # 2048
 activation='relu',
 kernel_regularizer=regularizers.l2(config.L2_REGULARIZATION),
 name='dense_2048'
 )(x)
 x = layers.BatchNormalization(name='bn_2')(x)
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout')(x)
 
 # Output layer (float32 for mixed precision)
 output = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
 
 model = models.Model(inputs=base_model.input, outputs=output, name=f'DenseNet121_V9_seed{seed}')
 
 # Print summary
 trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total = model.count_params()
 
 print(f" Total params: {total:,}")
 print(f" Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
 
 return model, base_model

print(" V9 Model builder defined")
print(f" Dense units: {config.DENSE_UNITS} (paper spec)")
print(f" Dropout: {config.DROPOUT_RATE}")
print(f" L2 regularization: {config.L2_REGULARIZATION}")

In [ ]:
# Learning Rate Warmup Scheduler

class WarmupCosineDecayScheduler(Callback):
 """
 V9: Learning Rate Warmup with Cosine Decay.
 
 Prevents the instability seen in V7 when unfreezing layers:
 1. Start with very low LR (warmup_start_lr)
 2. Linear warmup to target LR
 3. Cosine decay for rest of training
 """
 
 def __init__(self, target_lr, warmup_epochs, total_epochs, 
 warmup_start_lr=1e-7, min_lr=1e-8, verbose=True):
 super().__init__()
 self.target_lr = target_lr
 self.warmup_epochs = warmup_epochs
 self.total_epochs = total_epochs
 self.warmup_start_lr = warmup_start_lr
 self.min_lr = min_lr
 self.verbose = verbose
 
 def on_epoch_begin(self, epoch, logs=None):
 if epoch < self.warmup_epochs:
 # Linear warmup
 lr = self.warmup_start_lr + (self.target_lr - self.warmup_start_lr) * (epoch / self.warmup_epochs)
 else:
 # Cosine decay
 progress = (epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
 lr = self.min_lr + 0.5 * (self.target_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
 
 # TensorFlow 2.16+ compatible
 self.model.optimizer.learning_rate.assign(lr)
 
 if self.verbose and (epoch < self.warmup_epochs or epoch % 10 == 0):
 phase = "warmup" if epoch < self.warmup_epochs else "decay"
 print(f"\n Epoch {epoch+1}: LR = {lr:.2e} ({phase})")


print(" LR Warmup Scheduler defined")
print(f" Warmup epochs: {config.WARMUP_EPOCHS}")
print(f" Warmup start LR: {config.WARMUP_START_LR}")

In [ ]:
# V9 Callbacks

class DiagnosticCallback(Callback):
 """Lightweight diagnostic callback."""
 
 def __init__(self, check_interval=10):
 super().__init__()
 self.check_interval = check_interval
 
 def on_epoch_end(self, epoch, logs=None):
 if (epoch + 1) % self.check_interval == 0:
 print(f"\n{'='*50}")
 print(f" Epoch {epoch + 1} Checkpoint")
 print(f" Val AUC: {logs.get('val_auc', 0):.4f}")
 print(f" Val Loss: {logs.get('val_loss', 0):.4f}")
 print(f" Val Recall: {logs.get('val_recall', 0):.4f}")
 print(f"{'='*50}")


def get_callbacks(stage_name, config, model_idx=0, min_epochs=15, 
 use_warmup=False, target_lr=None, total_epochs=None):
 """Get V9 callbacks with optional warmup."""
 
 callbacks = [
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_auc.h5'),
 monitor='val_auc',
 mode='max',
 save_best_only=True,
 verbose=1
 ),
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_loss.h5'),
 monitor='val_loss',
 mode='min',
 save_best_only=True,
 verbose=0
 ),
 EarlyStopping(
 monitor='val_auc',
 mode='max',
 patience=config.EARLY_STOP_PATIENCE,
 restore_best_weights=True,
 verbose=1,
 start_from_epoch=min_epochs
 ),
 DiagnosticCallback(check_interval=10)
 ]
 
 # Add warmup scheduler for fine-tuning stages
 if use_warmup and target_lr and total_epochs:
 callbacks.append(WarmupCosineDecayScheduler(
 target_lr=target_lr,
 warmup_epochs=config.WARMUP_EPOCHS,
 total_epochs=total_epochs,
 warmup_start_lr=config.WARMUP_START_LR
 ))
 else:
 # Standard LR reduction for Stage 1
 callbacks.append(ReduceLROnPlateau(
 monitor='val_loss',
 factor=config.LR_REDUCE_FACTOR,
 patience=config.LR_REDUCE_PATIENCE,
 min_lr=1e-8,
 verbose=1
 ))
 
 return callbacks

print(" V9 Callbacks defined")

In [ ]:
# V9 Single Model Training Function

def train_single_model(model_idx, seed, config, train_dataset, val_dataset):
 """
 Train a single model using 3-stage progressive unfreezing.
 
 Stage 1: Head only (25 epochs)
 Stage 2: Last dense block with warmup (40 epochs)
 Stage 3: Partial fine-tune with warmup (80 epochs)
 """
 print("\n" + "="*70)
 print(f" TRAINING MODEL {model_idx + 1}/{config.NUM_ENSEMBLE_MODELS} (seed={seed})")
 print("="*70)
 
 # Build model
 model, base_model = build_model(config, seed=seed)
 
 # Get loss function
 if config.USE_FOCAL_LOSS:
 loss_fn = get_focal_loss(config.FOCAL_LOSS_GAMMA, config.FOCAL_LOSS_ALPHA)
 else:
 loss_fn = BinaryCrossentropy(label_smoothing=config.LABEL_SMOOTHING)
 
 metrics = [
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
 
 # STAGE 1: Train Classification Head Only
 print(f"\n STAGE 1: Training Head ({config.STAGE1_EPOCHS} epochs)")
 print(f" LR: {config.STAGE1_LR}, All backbone frozen")
 
 base_model.trainable = False
 
 optimizer = Adam(learning_rate=config.STAGE1_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history1 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE1_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage1', config, model_idx=model_idx, 
 min_epochs=config.MIN_EPOCHS_STAGE1),
 verbose=1
 )
 
 best_auc_s1 = max(history1.history['val_auc'])
 print(f" Stage 1 best val_auc: {best_auc_s1:.4f}")
 
 # STAGE 2: Unfreeze Last Dense Block (with warmup)
 print(f"\n STAGE 2: Last Dense Block ({config.STAGE2_EPOCHS} epochs)")
 print(f" LR: {config.STAGE2_LR} with {config.WARMUP_EPOCHS}-epoch warmup")
 
 # Unfreeze last dense block only
 base_model.trainable = True
 for layer in base_model.layers:
 layer.trainable = False
 
 # Unfreeze from conv5_block1 onwards
 unfreeze_from = 'conv5_block1'
 found = False
 for layer in base_model.layers:
 if unfreeze_from in layer.name or found:
 layer.trainable = True
 found = True
 
 trainable_count = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 print(f" Trainable params: {trainable_count:,}")
 
 # Start with warmup LR
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history2 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage2', config, model_idx=model_idx,
 min_epochs=config.MIN_EPOCHS_STAGE2,
 use_warmup=True, target_lr=config.STAGE2_LR,
 total_epochs=config.STAGE2_EPOCHS),
 verbose=1
 )
 
 best_auc_s2 = max(history2.history['val_auc'])
 print(f" Stage 2 best val_auc: {best_auc_s2:.4f}")
 
 # STAGE 3: Partial Fine-Tuning (with warmup)
 print(f"\n STAGE 3: Partial Fine-Tune ({config.STAGE3_EPOCHS} epochs)")
 print(f" LR: {config.STAGE3_LR} with {config.WARMUP_EPOCHS}-epoch warmup")
 print(f" Freezing first {config.FREEZE_STAGE3} layers")
 
 # Unfreeze more layers
 base_model.trainable = True
 for i, layer in enumerate(base_model.layers):
 layer.trainable = i >= config.FREEZE_STAGE3
 
 trainable_count = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total_count = model.count_params()
 print(f" Trainable params: {trainable_count:,} ({trainable_count/total_count*100:.1f}%)")
 
 # Start with warmup LR
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history3 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE3_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage3', config, model_idx=model_idx,
 min_epochs=config.MIN_EPOCHS_STAGE3,
 use_warmup=True, target_lr=config.STAGE3_LR,
 total_epochs=config.STAGE3_EPOCHS),
 verbose=1
 )
 
 best_auc_s3 = max(history3.history['val_auc'])
 print(f" Stage 3 best val_auc: {best_auc_s3:.4f}")
 
 # Load best weights
 best_path = config.CHECKPOINT_PATH / f'model{model_idx}_stage3_best_auc.h5'
 if best_path.exists():
 model.load_weights(str(best_path))
 print(f" Loaded best weights from {best_path.name}")
 
 # Summary
 print(f"\n Model {model_idx + 1} Training Summary:")
 print(f" Stage 1: {best_auc_s1:.4f}")
 print(f" Stage 2: {best_auc_s2:.4f}")
 print(f" Stage 3: {best_auc_s3:.4f}")
 print(f" Best: {max(best_auc_s1, best_auc_s2, best_auc_s3):.4f}")
 
 return model, {'stage1': history1, 'stage2': history2, 'stage3': history3}

print(" V9 Training function defined")

In [ ]:
# TRAIN ALL ENSEMBLE MODELS

print("\n" + "="*70)
print(" V9 ENSEMBLE TRAINING")
print(f" Training {config.NUM_ENSEMBLE_MODELS} models with seeds: {config.ENSEMBLE_SEEDS}")
total_epochs = (config.STAGE1_EPOCHS + config.STAGE2_EPOCHS + config.STAGE3_EPOCHS) * config.NUM_ENSEMBLE_MODELS
print(f" Total epochs: {total_epochs}")
print("="*70)

ensemble_models = []
ensemble_histories = []

for idx, seed in enumerate(config.ENSEMBLE_SEEDS):
 # Create fresh dataset with different shuffle for each model
 train_ds = create_dataset(train_images, train_labels, config, training=True, shuffle_seed=seed)
 val_ds = create_dataset(val_images, val_labels, config, training=False)
 
 model, histories = train_single_model(
 model_idx=idx,
 seed=seed,
 config=config,
 train_dataset=train_ds,
 val_dataset=val_ds
 )
 
 ensemble_models.append(model)
 ensemble_histories.append(histories)
 
 # Clear some memory
 gc.collect()

print("\n" + "="*70)
print(f" ALL {config.NUM_ENSEMBLE_MODELS} MODELS TRAINED")
print("="*70)

In [ ]:
# QUICK RESTART CELL (Run after kernel restart)
# If need to restart training or kernel crashed, run this cell
# to quickly reload cached data without re-running all cells.

# Uncomment below if needed:

# print(" Quick restart - Loading from cache...")
# 
# # Clear memory first
# import gc
# tf.keras.backend.clear_session()
# gc.collect()
# 
# # Reload cached data (fast!)
# train_cache = np.load(config.CACHE_PATH / 'train_cache_v9.npz')
# val_cache = np.load(config.CACHE_PATH / 'val_cache_v9.npz')
# test_cache = np.load(config.CACHE_PATH / 'test_cache_v9.npz')
# 
# train_images, train_labels = train_cache['images'], train_cache['labels']
# val_images, val_labels = val_cache['images'], val_cache['labels']
# test_images, test_labels = test_cache['images'], test_cache['labels']
# 
# print(f" Loaded from cache: Train={len(train_images)}, Val={len(val_images)}, Test={len(test_images)}")
# 
# # Recreate datasets
# train_dataset = create_dataset(train_images, train_labels, config, training=True)
# val_dataset = create_dataset(val_images, val_labels, config, training=False)
# test_dataset = create_dataset(test_images, test_labels, config, training=False)
# 
# # Reset ensemble lists
# ensemble_models = []
# ensemble_histories = []
# print(f" Ready for training!")

print(" Quick restart cell - uncomment and run if kernel restarts")

---
## Phase 3: Evaluation & Result Visualization
Following the paper: Breast Cancer Detection → Plot Results → Generate Report

In [ ]:
# Test-Time Augmentation (TTA)

def apply_tta_augmentation(images, aug_type):
 """
 Apply specific augmentation for TTA.
 
 Aug types:
 0: Original
 1: Horizontal flip
 2: Vertical flip 
 3: Both flips
 """
 if aug_type == 0:
 return images
 elif aug_type == 1:
 return np.flip(images, axis=2)
 elif aug_type == 2:
 return np.flip(images, axis=1)
 elif aug_type == 3:
 return np.flip(np.flip(images, axis=2), axis=1)
 return images


def predict_with_tta(model, images, n_augmentations=4, batch_size=32):
 """Make predictions with TTA."""
 all_preds = []
 
 for aug_idx in range(n_augmentations):
 aug_images = apply_tta_augmentation(images, aug_idx)
 preds = model.predict(aug_images, batch_size=batch_size, verbose=0)
 all_preds.append(preds.flatten())
 
 return np.mean(all_preds, axis=0)


def ensemble_predict_with_tta(models, images, config):
 """
 Make ensemble predictions with TTA.
 
 Total predictions averaged: num_models × n_augmentations
 """
 all_model_preds = []
 
 for idx, model in enumerate(models):
 print(f" Predicting with Model {idx + 1}/{len(models)}...")
 
 if config.USE_TTA:
 preds = predict_with_tta(model, images, config.TTA_AUGMENTATIONS, config.BATCH_SIZE)
 else:
 preds = model.predict(images, batch_size=config.BATCH_SIZE, verbose=0).flatten()
 
 all_model_preds.append(preds)
 
 # Average across all models
 ensemble_preds = np.mean(all_model_preds, axis=0)
 
 return ensemble_preds, all_model_preds

print(" TTA and Ensemble prediction functions defined")
print(f" TTA augmentations: {config.TTA_AUGMENTATIONS}")
print(f" Total predictions averaged: {config.NUM_ENSEMBLE_MODELS * config.TTA_AUGMENTATIONS}")

In [ ]:
# Reload Models (if needed after restart)

# Uncomment this cell if need to reload models after kernel restart

# print("\n" + "="*70)
# print(" RELOADING ENSEMBLE MODELS")
# print("="*70)

# ensemble_models = []
# for idx, seed in enumerate(config.ENSEMBLE_SEEDS):
# model_path = config.CHECKPOINT_PATH / f'model{idx}_stage3_best_auc.h5'
# if model_path.exists():
# model, _ = build_model(config, seed=seed)
# model.load_weights(str(model_path))
# ensemble_models.append(model)
# print(f" Loaded Model {idx + 1}")
# else:
# print(f" Model {idx + 1} not found")

# print(f"\n Loaded {len(ensemble_models)} models")

print(" Model reload cell ready (uncomment if needed)")

In [ ]:
# V9 FINAL EVALUATION

print("\n" + "="*70)
print(" V9 FINAL EVALUATION (Ensemble + TTA)")
print("="*70)

# Get ensemble predictions
print("\n Getting ensemble predictions with TTA...")
y_pred_ensemble, individual_preds = ensemble_predict_with_tta(ensemble_models, test_images, config)
y_true = test_labels

# Calculate AUC
auc = roc_auc_score(y_true, y_pred_ensemble)
print(f"\n Ensemble Test Set AUC-ROC: {auc:.4f}")

# Individual model AUCs
print(f"\n Individual Model AUCs:")
for idx, preds in enumerate(individual_preds):
 model_auc = roc_auc_score(y_true, preds)
 print(f" Model {idx + 1}: {model_auc:.4f}")

# Find optimal threshold
fpr, tpr, thresholds = roc_curve(y_true, y_pred_ensemble)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n Optimal Threshold (Youden's J): {optimal_threshold:.3f}")

# Threshold sweep
print("\n" + "="*70)
print(" THRESHOLD SWEEP ANALYSIS")
print("="*70)
print(f"\n{'Threshold':<12} {'Sensitivity':<14} {'Specificity':<14} {'NPV':<10} {'FNR':<10}")
print("-" * 60)

for thresh in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, optimal_threshold, 0.55, 0.60]:
 y_pred_t = (y_pred_ensemble >= thresh).astype(int)
 
 tp = ((y_pred_t == 1) & (y_true == 1)).sum()
 tn = ((y_pred_t == 0) & (y_true == 0)).sum()
 fp = ((y_pred_t == 1) & (y_true == 0)).sum()
 fn = ((y_pred_t == 0) & (y_true == 1)).sum()
 
 sens = tp / (tp + fn) if (tp + fn) > 0 else 0
 spec = tn / (tn + fp) if (tn + fp) > 0 else 0
 npv = tn / (tn + fn) if (tn + fn) > 0 else 0
 fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
 
 marker = " ← Youden's J" if abs(thresh - optimal_threshold) < 0.001 else ""
 print(f"{thresh:<12.3f} {sens*100:<13.1f}% {spec*100:<13.1f}% {npv*100:<9.1f}% {fnr*100:<9.1f}%{marker}")

print("-" * 60)

# Final metrics at optimal threshold
y_pred_opt = (y_pred_ensemble >= optimal_threshold).astype(int)

tp = ((y_pred_opt == 1) & (y_true == 1)).sum()
tn = ((y_pred_opt == 0) & (y_true == 0)).sum()
fp = ((y_pred_opt == 1) & (y_true == 0)).sum()
fn = ((y_pred_opt == 0) & (y_true == 1)).sum()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
fnr = fn / (fn + tp)

print(f"\n V9 Final Metrics @ threshold={optimal_threshold:.3f}:")
print(f" AUC-ROC: {auc:.4f} (target: 0.85)")
print(f" Accuracy: {accuracy*100:.1f}%")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 85%)")
print(f" Specificity: {specificity*100:.1f}% (target: 75%)")
print(f" PPV: {ppv*100:.1f}%")
print(f" NPV: {npv*100:.1f}%")
print(f" FNR: {fnr*100:.1f}%")

# Comparison with previous versions
print(f"\n V9 vs Previous Versions:")
print(f" {'Metric':<15} {'V4':<10} {'V6':<10} {'V7':<10} {'V9':<10}")
print(f" {'-'*55}")
print(f" {'AUC':<15} {'0.707':<10} {'0.723':<10} {'0.732':<10} {auc:.3f}")
print(f" {'Sensitivity':<15} {'58.3%':<10} {'63.3%':<10} {'69.7%':<10} {sensitivity*100:.1f}%")
print(f" {'Specificity':<15} {'76.2%':<10} {'69.8%':<10} {'66.0%':<10} {specificity*100:.1f}%")

print("\n" + "="*70)

In [ ]:
# Plot Results

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: ROC Curve Comparison
ax1 = axes[0, 0]
ax1.plot(fpr, tpr, 'b-', linewidth=3, label=f'V9 Ensemble (AUC = {auc:.3f})')

colors = ['orange', 'green', 'red']
for idx, preds in enumerate(individual_preds):
 fpr_i, tpr_i, _ = roc_curve(y_true, preds)
 model_auc = roc_auc_score(y_true, preds)
 ax1.plot(fpr_i, tpr_i, '--', color=colors[idx % len(colors)], linewidth=1.5, alpha=0.7,
 label=f'Model {idx+1} (AUC = {model_auc:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax1.scatter([1-specificity], [sensitivity], c='red', s=200, marker='*', 
 label=f'Operating Point', zorder=5)
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('V9 ROC Curve - Ensemble vs Individual Models')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Distribution
ax2 = axes[0, 1]
ax2.hist(y_pred_ensemble[y_true == 0], bins=30, alpha=0.6, label='Benign', color='blue', density=True)
ax2.hist(y_pred_ensemble[y_true == 1], bins=30, alpha=0.6, label='Malignant', color='red', density=True)
ax2.axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2, 
 label=f'Threshold = {optimal_threshold:.3f}')
ax2.set_xlabel('Predicted Probability')
ax2.set_ylabel('Density')
ax2.set_title('V9 Prediction Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Confusion Matrix
ax3 = axes[1, 0]
cm = confusion_matrix(y_true, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
ax3.set_xlabel('Predicted')
ax3.set_ylabel('Actual')
ax3.set_title(f'Confusion Matrix @ threshold={optimal_threshold:.3f}')

# Plot 4: Version Comparison
ax4 = axes[1, 1]
versions = ['V4', 'V6', 'V7', 'V9']
aucs = [0.707, 0.723, 0.732, auc]
sens = [0.583, 0.633, 0.697, sensitivity]
spec = [0.762, 0.698, 0.660, specificity]

x = np.arange(len(versions))
width = 0.25

bars1 = ax4.bar(x - width, aucs, width, label='AUC', color='blue', alpha=0.8)
bars2 = ax4.bar(x, sens, width, label='Sensitivity', color='green', alpha=0.8)
bars3 = ax4.bar(x + width, spec, width, label='Specificity', color='orange', alpha=0.8)

ax4.set_xlabel('Version')
ax4.set_ylabel('Score')
ax4.set_title('Performance Comparison Across Versions')
ax4.set_xticks(x)
ax4.set_xticklabels(versions)
ax4.legend()
ax4.set_ylim(0, 1)
ax4.axhline(y=0.85, color='red', linestyle='--', alpha=0.5, label='Target')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
 for bar in bars:
 height = bar.get_height()
 ax4.annotate(f'{height:.2f}',
 xy=(bar.get_x() + bar.get_width() / 2, height),
 xytext=(0, 3), textcoords="offset points",
 ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(config.RESULTS_PATH / 'v9_results.png', dpi=150)
plt.show()

print(f"\n Results saved to {config.RESULTS_PATH / 'v9_results.png'}")

In [ ]:
# Generate CSV Report & Save Results

# Save detailed results JSON
results = {
 "version": "V9_OPTIMAL",
 "test_auc": float(auc),
 "optimal_threshold": float(optimal_threshold),
 "accuracy": float(accuracy),
 "sensitivity": float(sensitivity),
 "specificity": float(specificity),
 "ppv": float(ppv),
 "npv": float(npv),
 "fnr": float(fnr),
 "individual_model_aucs": [float(roc_auc_score(y_true, p)) for p in individual_preds],
 "comparison": {
 "v4": {"auc": 0.707, "sensitivity": 0.583, "specificity": 0.762},
 "v6": {"auc": 0.723, "sensitivity": 0.633, "specificity": 0.698},
 "v7": {"auc": 0.732, "sensitivity": 0.697, "specificity": 0.660},
 },
 "improvements_vs_v7": {
 "auc_change": float(auc - 0.732),
 "sensitivity_change": float(sensitivity - 0.697),
 "specificity_change": float(specificity - 0.660)
 },
 "configuration": {
 "img_size": list(config.IMG_SIZE),
 "dense_units": config.DENSE_UNITS,
 "use_clahe": config.USE_CLAHE,
 "clahe_clip_limit": config.CLAHE_CLIP_LIMIT,
 "num_ensemble_models": config.NUM_ENSEMBLE_MODELS,
 "use_tta": config.USE_TTA,
 "tta_augmentations": config.TTA_AUGMENTATIONS,
 "use_lr_warmup": config.USE_LR_WARMUP,
 "warmup_epochs": config.WARMUP_EPOCHS,
 "dropout_rate": config.DROPOUT_RATE,
 "l2_regularization": config.L2_REGULARIZATION,
 "class_weight_multiplier": config.MALIGNANT_WEIGHT_MULTIPLIER,
 "focal_loss_alpha": config.FOCAL_LOSS_ALPHA,
 "focal_loss_gamma": config.FOCAL_LOSS_GAMMA
 },
 "v9_evidence_based_changes": [
 "Dense units: 1024 → 2048 (paper spec)",
 "CLAHE preprocessing (mammography contrast enhancement)",
 f"LR warmup ({config.WARMUP_EPOCHS} epochs) - prevents V7 instability",
 f"Balanced dropout ({config.DROPOUT_RATE}) - V7's 0.0 caused instability",
 f"Class weight {config.MALIGNANT_WEIGHT_MULTIPLIER}x (between V6=2x, V7=4x)",
 f"3-model ensemble + {config.TTA_AUGMENTATIONS}-way TTA",
 "224×224 resolution (paper spec, proven stable)"
 ]
}

results_file = config.RESULTS_PATH / 'v9_results.json'
with open(results_file, 'w') as f:
 json.dump(results, f, indent=2)

# Generate CSV report
csv_data = {
 'Metric': ['AUC-ROC', 'Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'FNR'],
 'Value': [auc, accuracy, sensitivity, specificity, ppv, npv, fnr],
 'Target': [0.85, 0.90, 0.85, 0.75, 0.80, 0.85, 0.15],
 'V7_Baseline': [0.732, None, 0.697, 0.660, None, 0.759, 0.303]
}
csv_df = pd.DataFrame(csv_data)
csv_file = config.RESULTS_PATH / 'v9_metrics.csv'
csv_df.to_csv(csv_file, index=False)

print("\n" + "="*70)
print(" V9 OPTIMAL RESULTS SUMMARY")
print("="*70)
print(f"\n Target vs Achieved:")
print(f" AUC: {auc:.4f} (target: 0.85) {'' if auc >= 0.85 else ''}")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 85%) {'' if sensitivity >= 0.85 else ''}")
print(f" Specificity: {specificity*100:.1f}% (target: 75%) {'' if specificity >= 0.75 else ''}")

print(f"\n Improvement vs V7 Baseline:")
print(f" AUC: {'+' if auc > 0.732 else ''}{(auc-0.732)*100:.2f}%")
print(f" Sensitivity: {'+' if sensitivity > 0.697 else ''}{(sensitivity-0.697)*100:.1f} points")
print(f" Specificity: {'+' if specificity > 0.660 else ''}{(specificity-0.660)*100:.1f} points")

print(f"\n V9 Evidence-Based Improvements:")
for imp in results['v9_evidence_based_changes']:
 print(f" {imp}")

print(f"\n Results saved to:")
print(f" JSON: {results_file}")
print(f" CSV: {csv_file}")
print(f" Plot: {config.RESULTS_PATH / 'v9_results.png'}")
print("="*70)

---
## V9 OPTIMAL Training Notes

### Evidence-Based Configuration Choices

| Parameter | V4 | V6 | V7 | V9 | Rationale |
|-----------|----|----|----|----|----------|
| Dense Units | 1024 | 1024 | 1024 | **2048** | Paper spec |
| Stage3 LR | 1e-6 | 1e-5 | 1e-4 | **1e-4** | V7's value worked |
| Dropout | 0.4 | 0.3 | 0.0 | **0.25** | V7's 0.0 caused instability |
| Class Weight | OFF | 2.0x | 4.0x | **2.5x** | Balanced |
| L2 Reg | 0.01 | 0.001 | 5e-4 | **1e-4** | Light regularization |
| LR Warmup | No | No | No | **Yes** | Prevents instability |
| CLAHE | No | No | No | **Yes** | Mammography enhancement |
| Ensemble | No | No | No | **3 models** | Reduces variance |
| TTA | No | No | No | **4-way** | Improves robustness |

### What Worked in Previous Versions
- Higher Stage 2b/3 learning rate (V6→V7)
- Class weights enabled (V6→V7)
- Reduced regularization (V4→V6→V7)
- Extended training epochs (V6→V7)
- More unfrozen layers (V4→V6→V7)

### What Did NOT Work
- Zero dropout (V7) - caused training instability
- 4x class weight (V7) - too aggressive, hurt specificity
- 384×384 resolution (V8) - GPU memory issues
- No warmup when unfreezing (V7) - validation drops

### Expected Performance
Based on the analysis:
- **AUC**: 0.78-0.85 (V7 was 0.732)
- **Sensitivity**: 75-85% (V7 was 69.7%)
- **Specificity**: 70-78% (V7 was 66.0%)

### If Results Still Below Target
Consider:
1. EfficientNetB4 backbone (19M params vs 7M)
2. Multi-view fusion (CC + MLO together)
3. Attention mechanisms
4. External data augmentation (MixUp, CutMix)
5. Semi-supervised learning with unlabeled mammograms